# Bronze Layer - Data Model Ingestion

This notebook reads an Excel file containing data model sheets and ingests them into the bronze layer of our lakehouse architecture.

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Configuration
file_path = "/Volumes/workspace/default/data_modeling/Data model.xlsx"
bronze_catalog = "workspace"
bronze_schema = "bronze"

# List all sheets in the Excel file
sheets_df = spark.read.format("excel") \
    .option("operation", "listSheets") \
    .load(file_path)

sheet_names = [row.sheetName for row in sheets_df.collect()]
print(f"Found {len(sheet_names)} sheets in the Excel file:")
for sheet in sheet_names:
    print(f"  - {sheet}")


Found 7 sheets in the Excel file:
  - factSalesTable
  - dimCustomerTable
  - dimDateTable
  - dimRegionTable
  - dimProductTable
  - dimProductSubcategoryTable
  - dimProductCategoryTable


In [0]:
import pandas as pd

# Read and display each sheet
for sheet_name in sheet_names:
    print(f"\n{'='*80}")
    print(f"Sheet: {sheet_name}")
    print(f"{'='*80}")
    
    # Read the sheet using pandas
    pdf = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)
    
    # Display schema
    print(f"\nSchema:")
    df.printSchema()
    
    # Display row count
    row_count = df.count()
    print(f"\nRow count: {row_count}")
    
    # Display sample data
    print(f"\nSample data (first 5 rows):")
    display(df.limit(5))


Sheet: factSalesTable

Schema:
root
 |-- ProductKey: long (nullable = true)
 |-- OrderDateKey: long (nullable = true)
 |-- CustomerKey: long (nullable = true)
 |-- Gender: double (nullable = true)
 |-- OrderNumber: long (nullable = true)
 |-- OrderQuantity: long (nullable = true)
 |-- List Price: double (nullable = true)
 |-- Product Cost: double (nullable = true)


Row count: 114390

Sample data (first 5 rows):


ProductKey,OrderDateKey,CustomerKey,Gender,OrderNumber,OrderQuantity,List Price,Product Cost
344,20050722,11000,null,20061722,22,3399.99,1912.1544
353,20070722,11000,null,20081722,22,2319.99,1265.6195
485,20070722,11000,null,20081722,22,21.98,8.2205
530,20071104,11000,null,20082104,4,4.99,1.8663
214,20071104,11000,null,20082104,4,34.99,13.0863



Sheet: dimCustomerTable

Schema:
root
 |-- GeographyKey: long (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- CustomerKey: long (nullable = true)
 |-- CustomerName: string (nullable = true)


Row count: 9999

Sample data (first 5 rows):


GeographyKey,MaritalStatus,Gender,CustomerKey,CustomerName
612,M,M,11254,Customer1
612,M,M,11255,Customer2
612,M,M,11282,Customer3
612,M,M,11321,Customer4
612,M,M,11633,Customer5



Sheet: dimDateTable

Schema:
root
 |-- DateKey: long (nullable = true)
 |-- FullDateAlternateKey: timestamp (nullable = true)
 |-- EnglishMonthName: string (nullable = true)
 |-- CalendarYear: long (nullable = true)


Row count: 1188

Sample data (first 5 rows):


...

ProductKey,ProductSubcategoryKey,Color,StandardCost,ListPrice,Product Name
212,31,Red,12.0278,33.6442,Product 1
213,31,Red,13.8782,33.6442,Product 2
214,31,Red,13.0863,34.99,Product 3
218,23,White,3.3963,9.5,Product 4
219,23,White,3.3963,9.5,Product 5



Sheet: dimProductSubcategoryTable

Schema:
root
 |-- ProductSubcategoryKey: long (nullable = true)
 |-- ProductSubcategoryName: string (nullable = true)
 |-- ProductCategoryKey: long (nullable = true)


Row count: 40

Sample data (first 5 rows):


ProductSubcategoryKey,ProductSubcategoryName,ProductCategoryKey
1,Mountain Bikes,1
2,Road Bikes,1
3,Touring Bikes,1
4,Handlebars,2
5,Bottom Brackets,2



Sheet: dimProductCategoryTable

Schema:
root
 |-- ProductCategoryKey: long (nullable = true)
 |-- ProductCategoryName: string (nullable = true)


Row count: 5

Sample data (first 5 rows):


ProductCategoryKey,ProductCategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories
5,Food


In [0]:
# Ensure bronze schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{bronze_schema}")
print(f"Schema {bronze_catalog}.{bronze_schema} is ready.\n")

# Save each sheet as a table in bronze layer
for sheet_name in sheet_names:
    print(f"Processing {sheet_name}...")
    
    # Read the sheet using pandas
    pdf = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
    
    # Clean column names - replace spaces with underscores
    pdf.columns = pdf.columns.str.replace(' ', '_')
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)
    
    # Create table name (convert to lowercase for consistency)
    table_name = sheet_name.lower()
    full_table_name = f"{bronze_catalog}.{bronze_schema}.{table_name}"
    
    # Write to bronze layer as a Delta table
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)
    
    row_count = spark.table(full_table_name).count()
    print(f"  ✓ Saved to {full_table_name} ({row_count} rows)\n")

print("All sheets have been successfully saved to the bronze layer!")

Schema workspace.bronze is ready.

Processing factSalesTable...
  ✓ Saved to workspace.bronze.factsalestable (114390 rows)

Processing dimCustomerTable...
  ✓ Saved to workspace.bronze.dimcustomertable (9999 rows)

Processing dimDateTable...
  ✓ Saved to workspace.bronze.dimdatetable (1188 rows)

Processing dimRegionTable...
  ✓ Saved to workspace.bronze.dimregiontable (655 rows)

Processing dimProductTable...
  ✓ Saved to workspace.bronze.dimproducttable (400 rows)

Processing dimProductSubcategoryTable...
  ✓ Saved to workspace.bronze.dimproductsubcategorytable (40 rows)

Processing dimProductCategoryTable...
  ✓ Saved to workspace.bronze.dimproductcategorytable (5 rows)

All sheets have been successfully saved to the bronze layer!
